In [1]:
# 16. Tokenization and Label Alignment
from IPython.display import display
import numpy as np
import pandas as pd

from baseline_ner import LAB_DIR, SEED, load_conll_data

train_rows, test_rows, label_names = load_conll_data()
test_words = [row["tokens"] for row in test_rows]
outside_id = label_names.index("O")

import torch
from transformers import AutoConfig, AutoTokenizer

CHECKPOINT = "google/bert_uncased_L-2_H-128_A-2"
IGNORE_INDEX = -100
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, use_fast=True)
MAX_LENGTH = AutoConfig.from_pretrained(CHECKPOINT).max_position_embeddings


def first_subtoken_mask(word_ids):
    return [
        word_id is not None and (i == 0 or word_id != word_ids[i - 1])
        for i, word_id in enumerate(word_ids)
    ]


def encode_words(sentences, tags=None):
    encoded = tokenizer(
        sentences,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors="pt",
    )
    word_ids = [encoded.word_ids(i) for i in range(len(sentences))]
    masks = [first_subtoken_mask(ids) for ids in word_ids]
    for words, ids, mask in zip(sentences, word_ids, masks, strict=True):
        if [idx for idx, is_first in zip(ids, mask) if is_first] != list(range(len(words))):
            raise ValueError("A sentence lost word alignment during tokenization.")
    if tags is not None:
        encoded["labels"] = torch.tensor([
            [sentence_tags[idx] if is_first else IGNORE_INDEX for idx, is_first in zip(ids, mask)]
            for sentence_tags, ids, mask in zip(tags, word_ids, masks, strict=True)
        ])
    return encoded, word_ids, masks


sample_rows = [train_rows[3], train_rows[1]]
sample_words = [row["tokens"] for row in sample_rows]
sample_tags = [row["ner_tags"] for row in sample_rows]
encoded, word_ids, masks = encode_words(sample_words, sample_tags)

rows = []
for sentence_id, words in enumerate(sample_words):
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][sentence_id].tolist())
    for pos, token in enumerate(tokens):
        word_id = word_ids[sentence_id][pos]
        label_id = encoded["labels"][sentence_id, pos].item()
        rows.append({
            "Sentence": sentence_id + 1,
            "Token": token,
            "Input ID": encoded["input_ids"][sentence_id, pos].item(),
            "Attention mask": encoded["attention_mask"][sentence_id, pos].item(),
            "Word ID": word_id,
            "Word": words[word_id] if word_id is not None else "",
            "Aligned label ID": label_id,
            "BIO": label_names[label_id] if label_id != IGNORE_INDEX else "",
        })
alignment_table = pd.DataFrame(rows)
alignment_table["Word ID"] = alignment_table["Word ID"].astype("Int64")
print("Strategy A | first subtoken | ignore index = -100")
with pd.option_context("display.max_rows", None):
    display(alignment_table)

C:\Users\Acer\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Strategy A | first subtoken | ignore index = -100


,Sentence,Token,Input ID,Attention mask,Word ID,Word,Aligned label ID,BIO
0,1,[CLS],101,1,<NA>,,-100,
1,1,the,1996,1,0,The,0,O
2,1,european,2647,1,1,European,3,B-ORG
3,1,commission,3222,1,2,Commission,4,I-ORG
4,1,said,2056,1,3,said,0,O
5,1,on,2006,1,4,on,0,O
6,1,thursday,9432,1,5,Thursday,0,O
7,1,it,2009,1,6,it,0,O
8,1,disagreed,18335,1,7,disagreed,0,O
9,1,with,2007,1,8,with,0,O


In [2]:
# 17. Experiment 1 - Tokenization Analysis
from collections import Counter

rng = np.random.default_rng(SEED)
sentence_groups = {
    "Short (<=10 words)": [i for i, words in enumerate(test_words) if len(words) <= 10],
    "Long (>=20 words)": [i for i, words in enumerate(test_words) if len(words) >= 20],
}
sample = [
    (group, int(index))
    for group, candidates in sentence_groups.items()
    for index in rng.choice(candidates, size=10, replace=False)
]

sample_encoding = tokenizer(
    [test_words[index] for _, index in sample],
    is_split_into_words=True,
    add_special_tokens=False,
)
word_rows = []
for batch_index, (group, index) in enumerate(sample):
    counts = Counter(sample_encoding.word_ids(batch_index))
    for word_id, label_id in enumerate(test_rows[index]["ner_tags"]):
        pieces = counts[word_id]
        word_rows.append({
            "Sentence ID": index,
            "Length group": group,
            "Word group": "Entity" if label_id != outside_id else "Non-entity",
            "Tokens": pieces,
            "Subword splits": max(0, pieces - 1),
            "Split words": int(pieces > 1),
        })

word_stats = pd.DataFrame(word_rows)


def summarize_tokenization(keys):
    summary = (
        word_stats.groupby(keys, sort=False)
        .agg(
            Words=("Tokens", "size"),
            Tokens=("Tokens", "sum"),
            **{"Subword splits": ("Subword splits", "sum"), "Split words": ("Split words", "sum")},
        )
        .reset_index()
    )
    summary["R"] = (summary["Tokens"] / summary["Words"]).round(4)
    summary["Split rate (%)"] = (100 * summary["Split words"] / summary["Words"]).round(2)
    return summary


sentence_stats = summarize_tokenization(["Sentence ID", "Length group"])
sentence_stats["Sentence"] = [" ".join(test_words[i]) for i in sentence_stats["Sentence ID"]]
with pd.option_context("display.max_colwidth", None):
    display(sentence_stats)
display(summarize_tokenization("Length group"))
display(summarize_tokenization("Word group"))

,Sentence ID,Length group,Words,Tokens,Subword splits,Split words,R,Split rate (%),Sentence
0,281,Short (<=10 words),10,20,10,5,2.0000,50.00,Fall of wickets : 1-59 2-70 3-78 4-90 5-160 .
1,2882,Short (<=10 words),2,4,2,1,2.0000,50.00,0-1 .
2,294,Short (<=10 words),6,6,0,0,1.0000,0.00,CRICKET - SHEFFIELD SHIELD SCORE .
3,2546,Short (<=10 words),7,14,7,4,2.0000,57.14,14. Hilde Gerg ( Germany ) 1:49.84
4,1683,Short (<=10 words),3,3,0,0,1.0000,0.00,Trade slow .
5,1655,Short (<=10 words),3,5,2,1,1.6667,33.33,Prudential Securities Incorporated
6,2679,Short (<=10 words),8,10,2,1,1.2500,12.50,SKI JUMPING-LEADING WORLD CUP RESULTS / STANDINGS .
7,331,Short (<=10 words),2,2,0,0,1.0000,0.00,Group D
8,593,Short (<=10 words),6,6,0,0,1.0000,0.00,soccer matches played on Friday :
9,3069,Short (<=10 words),5,5,0,0,1.0000,0.00,Ross County 4 Montrose 4


,Length group,Words,Tokens,Subword splits,Split words,R,Split rate (%)
0,Short (<=10 words),52,75,23,12,1.4423,23.08
1,Long (>=20 words),319,362,43,27,1.1348,8.46


,Word group,Words,Tokens,Subword splits,Split words,R,Split rate (%)
0,Non-entity,312,348,36,21,1.1154,6.73
1,Entity,59,89,30,18,1.5085,30.51
